## Configurando ambiente

In [1]:
%pip install pydantic[email]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 17.0 MB/s eta 0:00:00


## Validação básica por Regras

In [2]:
from enum import auto, IntFlag
from typing import Any
import hashlib
import re
import enum

from pydantic import (
    BaseModel,
    EmailStr,
    Field,
    SecretStr,
    ValidationError,
)


class Role(IntFlag):
    Salerman = auto()
    Client = auto()
    Admin = Salerman | Client


class User(BaseModel):
    username: str = Field(examples=["LíbnaRaffaely"])


    email: EmailStr = Field(
        examples=["libnaRaf@teste.com"],
        description="The email address of the user",
        frozen=True,
    )

    password: SecretStr = Field(
        examples=["Password123"], description="The password of the user"
    )

    role: Role = Field( description="The role of the user")


# função de validação
def validate(data: dict[str, Any]) -> None:
    try:
        # esse .model_validate vem do pydantic e essa classe criata o possui pois definimos ela assim User(BaseModel)
        user = User.model_validate(data)
        print(user)
    except ValidationError as e:
        print("User is invalid")
        for error in e.errors():
            print(error)



def main() -> None:
    good_data = {
        "username": "libnadejesus",
        "email": "Libna@test.com",
        "password": "Password123#",
        "role": Role.Salerman,
    }

    good_data2 = {
        "username": "libna de jesus",
        "email": "Libna@test.com",
        "password": "Password123#",
        "role": Role(2),
    }



    bad_data = {"email": "lib1# ssxd", "password": "e q 2 #"}

    bad_data2 = {
        "username": "libna dejesus!",
        "email": "Libna@test.com",
        "password": "Password123#",
        "role": "Client",
    }


    print("Estrutura de dados Corretos: ")

    validate(good_data)
    print()

    validate(good_data2)
    print()



    print("Estrutura de dados Incorretos:")
    validate(bad_data)
    print()
    validate(bad_data2)

if __name__ == "__main__":
    main()



Estrutura de dados Corretos: 
username='libnadejesus' email='Libna@test.com' password=SecretStr('**********') role=<Role.Salerman: 1>

username='libna de jesus' email='Libna@test.com' password=SecretStr('**********') role=<Role.Client: 2>

Estrutura de dados Incorretos:
User is invalid
{'type': 'missing', 'loc': ('username',), 'msg': 'Field required', 'input': {'email': 'lib1# ssxd', 'password': 'e q 2 #'}, 'url': 'https://errors.pydantic.dev/2.12/v/missing'}
{'type': 'value_error', 'loc': ('email',), 'msg': 'value is not a valid email address: An email address must have an @-sign.', 'input': 'lib1# ssxd', 'ctx': {'reason': 'An email address must have an @-sign.'}}
{'type': 'missing', 'loc': ('role',), 'msg': 'Field required', 'input': {'email': 'lib1# ssxd', 'password': 'e q 2 #'}, 'url': 'https://errors.pydantic.dev/2.12/v/missing'}

User is invalid
{'type': 'enum', 'loc': ('role',), 'msg': 'Input should be 1, 2 or 3', 'input': 'Client', 'ctx': {'expected': '1, 2 or 3'}, 'url': 'http

Acima foi aplicado testes de formatos de dados enviados na string, para o pydantic fazer a validação.

A validação ocorre pela chamada de uma função do pydantic a ***.model_validate***, que foi possivel chamar pela definição da classe como: **class User(BaseModel):**

- Com base nas annotations ao definir cada atributo da classe é estabelecido padrões de validações:
  - EmailStr
  - SecretStr
  - str
  -  Field (para estabelecer descrição do formato e valor padrão/default)



## Validação Field_validator

Proposta de validação de dados para aluguel de carro


In [13]:
from enum import auto, IntFlag
from typing import Any
import hashlib
import re
import enum

from pydantic import (
    BaseModel,
    EmailStr,
    Field,
    SecretStr,
    ValidationError,
    field_validator,
    model_validator,
)

VALID_PLATE_REGEX = re.compile(r"^[A-Z]{3}[0-9][A-Z0-9][0-9]{2}$")
VALID_NAME_REGEX = re.compile(r"^[a-zA-Z]{2,}$")

class TypeCar(enum.IntFlag):
    Hatch = 1
    Sedan = 2
    Suv = 3


class Rent(BaseModel):
    username: str = Field(examples=["LíbnaRaffaely"])


    email: EmailStr = Field(
        examples=["libnaRaf@teste.com"],
        description="The email address of the user",
        frozen=True,
    )

    plateCar: str = Field(description="Plate of car")

    typeCar: TypeCar = Field( description="type of car for rent")

    @field_validator("username")
    @classmethod
    def validate_username(cls, value: str) -> str:
      if not VALID_NAME_REGEX.match(value):
        raise ValueError("Invalid username")
      return value


    @field_validator("plateCar")
    @classmethod
    def validate_plate(cls, value: str) -> str:
      if not VALID_PLATE_REGEX.match(value):
        raise ValueError("Invalid plate")
      return value

    @field_validator("typeCar", mode = "before")
    @classmethod
    def validate_type_car(cls, value: int | str | TypeCar) -> TypeCar:
      op = {int: lambda x: TypeCar(x), str: lambda x: TypeCar[x], TypeCar: lambda x: x}
      try:
        return op[type(value)](value)
      except (KeyError, ValueError):
        raise ValueError(
          f"Role is invalid, please use one of the following: {', '.join([x.name for x in TypeCar])}" # type: ignore
        )

    # o model_validator é util para  validar regras de negócio que interdependem 2 atributos da classe
    #para teste criaremos a  seguinte regra:
    """ Dependendo do tipo de veículo a placa deverá ter uma letra definida:
    - Hatch: A
    - Sedan: B
    - Suv: C
    """
    @model_validator(mode="before")
    @classmethod
    def validate_rent(cls, value: dict[str, Any]) -> dict[str, Any]:
      if value["typeCar"] == TypeCar.Hatch  and value["plateCar"][0]!= "A":
        raise ValueError("Invalid plate Hatch")
      if value["typeCar"] == TypeCar.Sedan  and value["plateCar"][0] != "B":
        raise ValueError("Invalid plate Sedan")
      if value["typeCar"] == TypeCar.Suv  and value["plateCar"][0] != "C":
        raise ValueError("Invalid plate Suv")

      return value


    # função de validação
    def validate(data: dict[str, Any]) -> None:
        try:
            rent = Rent.model_validate(data)
            print(rent)
        except ValidationError as e:
            print("User is invalid")
            for error in e.errors():
                print(error)



def main() -> None:
  test_data = dict(
      good_data={
        "username": "libnadejesus",
        "email": "Libna@test.com",
        "plateCar": "ABC1234",
        "typeCar": TypeCar.Hatch,
      },

    bad_data = {
        "username": "libnadejesus25",
        "email": "Libna@test.com",
        "plateCar": "BBC123425",
        "typeCar": TypeCar.Sedan,
    },

    bad_data1 = {
        "username": "libnadejesus",
        "email": "Libna@test.com",
        "plateCar": "ABC1234",
        "typeCar": TypeCar.Sedan,
    }

  )


  for example_name, data in test_data.items():
    print(example_name)
    validate(data)
    print()

if __name__ == "__main__":
    main()



good_data
username='libnadejesus' email='Libna@test.com' plateCar='ABC1234' typeCar=<TypeCar.Hatch: 1>

bad_data
User is invalid
{'type': 'value_error', 'loc': ('username',), 'msg': 'Value error, Invalid username', 'input': 'libnadejesus25', 'ctx': {'error': ValueError('Invalid username')}, 'url': 'https://errors.pydantic.dev/2.12/v/value_error'}
{'type': 'value_error', 'loc': ('plateCar',), 'msg': 'Value error, Invalid plate', 'input': 'BBC123425', 'ctx': {'error': ValueError('Invalid plate')}, 'url': 'https://errors.pydantic.dev/2.12/v/value_error'}

bad_data1
User is invalid
{'type': 'value_error', 'loc': (), 'msg': 'Value error, Invalid plate Sedan', 'input': {'username': 'libnadejesus', 'email': 'Libna@test.com', 'plateCar': 'ABC1234', 'typeCar': <TypeCar.Sedan: 2>}, 'ctx': {'error': ValueError('Invalid plate Sedan')}, 'url': 'https://errors.pydantic.dev/2.12/v/value_error'}



## Serialização - model_serializer

In [14]:
from enum import auto, IntFlag
from typing import Any
import hashlib
import re
import enum

from pydantic import (
    BaseModel,
    EmailStr,
    Field,
    SecretStr,
    ValidationError,
    field_validator,
    model_validator,
    field_serializer,
    model_serializer,
)

VALID_PLATE_REGEX = re.compile(r"^[A-Z]{3}[0-9][A-Z0-9][0-9]{2}$")
VALID_NAME_REGEX = re.compile(r"^[a-zA-Z]{2,}$")

class TypeCar(enum.IntFlag):
    Hatch = 1
    Sedan = 2
    Suv = 3


class Rent(BaseModel):
    username: str = Field(examples=["LíbnaRaffaely"])


    email: EmailStr = Field(
        examples=["libnaRaf@teste.com"],
        description="The email address of the user",
        frozen=True,
    )

    plateCar: str = Field(description="Plate of car")

    typeCar: TypeCar = Field( description="type of car for rent")

    @field_validator("username")
    @classmethod
    def validate_username(cls, value: str) -> str:
      if not VALID_NAME_REGEX.match(value):
        raise ValueError("Invalid username")
      return value


    @field_validator("plateCar")
    @classmethod
    def validate_plate(cls, value: str) -> str:
      if not VALID_PLATE_REGEX.match(value):
        raise ValueError("Invalid plate")
      return value

    @field_validator("typeCar", mode = "before")
    @classmethod
    def validate_type_car(cls, value: int | str | TypeCar) -> TypeCar:
      op = {int: lambda x: TypeCar(x), str: lambda x: TypeCar[x], TypeCar: lambda x: x}
      try:
        return op[type(value)](value)
      except (KeyError, ValueError):
        raise ValueError(
          f"Role is invalid, please use one of the following: {', '.join([x.name for x in TypeCar])}" # type: ignore
        )


    @model_validator(mode="before")
    @classmethod
    def validate_rent(cls, value: dict[str, Any]) -> dict[str, Any]:
      if value["typeCar"] == TypeCar.Hatch  and value["plateCar"][0]!= "A":
        raise ValueError("Invalid plate Hatch")
      if value["typeCar"] == TypeCar.Sedan  and value["plateCar"][0] != "B":
        raise ValueError("Invalid plate Sedan")
      if value["typeCar"] == TypeCar.Suv  and value["plateCar"][0] != "C":
        raise ValueError("Invalid plate Suv")

      return value

      #Novas implementações de serialização

      @field_serializer("typeCar")
      @classmethod
      def serialize_typeCar(cls, value) -> str:
        return value.name

      @model_serializer(mode="wrap", when_used="json")
      def serialize_rent(self, serializer, info) -> dict[str, Any]:
        if not info.include and not info.exclude:
            return {"plate": self.plateCar, "typeCar": self.typeCar.name}
        return serializer(self)

    # função de validação
    def validate(data: dict[str, Any]) -> None:
        try:
            rent = Rent.model_validate(data)
            print(rent)
        except ValidationError as e:
            print("User is invalid")
            for error in e.errors():
                print(error)



def main() -> None:
  good_data={
        "username": "libnadejesus",
        "email": "Libna@test.com",
        "plateCar": "ABC1234",
        "typeCar": TypeCar.Hatch,
      }
  rent = Rent.model_validate(good_data)

  if rent:
    print(
      "The serializer that returns a dict:",
      rent.model_dump(),
      sep="\n",
      end="\n\n",
    )
    print(
      "The serializer that returns a JSON string:",
      rent.model_dump(mode="json"),
      sep="\n",
      end="\n\n",
    )
    print(
      "The serializer that returns a json string, excluding the typeCar:",
      rent.model_dump(exclude=["typeCar"], mode="json"), # type: ignore
      sep="\n",
      end="\n\n",
    )
    print("The serializer that encodes all values to a dict:", dict(rent), sep="\n")



if __name__ == "__main__":
    main()



The serializer that returns a dict:
{'username': 'libnadejesus', 'email': 'Libna@test.com', 'plateCar': 'ABC1234', 'typeCar': <TypeCar.Hatch: 1>}

The serializer that returns a JSON string:
{'username': 'libnadejesus', 'email': 'Libna@test.com', 'plateCar': 'ABC1234', 'typeCar': 1}

The serializer that returns a json string, excluding the typeCar:
{'username': 'libnadejesus', 'email': 'Libna@test.com', 'plateCar': 'ABC1234'}

The serializer that encodes all values to a dict:
{'username': 'libnadejesus', 'email': 'Libna@test.com', 'plateCar': 'ABC1234', 'typeCar': <TypeCar.Hatch: 1>}
